In [1]:
# load R packages 
options(warn=-1)
library(psych)
library(GPArotation)
library(nFactors)
library(polycor)
library(lavaan)
#library("parameters")

Loading required package: MASS
Loading required package: boot

Attaching package: ‘boot’

The following object is masked from ‘package:psych’:

    logit

Loading required package: lattice

Attaching package: ‘lattice’

The following object is masked from ‘package:boot’:

    melanoma


Attaching package: ‘nFactors’

The following object is masked from ‘package:lattice’:

    parallel


Attaching package: ‘polycor’

The following object is masked from ‘package:psych’:

    polyserial

This is lavaan 0.6-5
lavaan is BETA software! Please report any bugs.

Attaching package: ‘lavaan’

The following object is masked from ‘package:psych’:

    cor2cov



In [5]:
# Load loadings from Gagne 2019 and common qns data from predator task

clinical_study = 1  # 0 for confirmatory_study

if (clinical_study == 1) {
    Gagne_loadings = read.csv('./supplementary_data/factor_analysis/CommonLoadings_ClinicalData_Gagne2019.csv', sep=',')
   } else {
    Gagne_loadings = read.csv('./supplementary_data/factor_analysis/CommonLoadings_ConfAnalysis_Gagne2019.csv', sep=',')
}

Qns_data_common_predator = read.csv('./supplementary_data/factor_analysis/qnsdata_predator_CommonItemsWithGagne2019.csv', sep=',')

head(Qns_data_common_predator)
nrow(Qns_data_common_predator)
ncol(Qns_data_common_predator)
(Gagne_loadings)


X,AI02_01,AI02_02,AI02_03,AI02_04,AI02_05,AI02_06,AI02_08,AI02_09,AI02_10,...,PW01_04,PW01_07,PW01_08,PW01_09,PW01_10,PW01_12,PW01_13,PW01_15,PW01_16,REF
0,2,3,3,4,2,3,3,2,3,...,4,2,4,3,5,2,3,2,3,SinglePredator_LessInstructions_63e590489911159364dcf70d
1,3,2,3,3,2,3,2,2,3,...,4,4,3,3,5,5,4,4,3,SinglePredator_LessInstructions_5ea3325a58e7b70dd3026a42
2,2,2,2,3,2,3,2,3,2,...,3,3,5,2,5,4,3,3,3,SinglePredator_LessInstructions_5e88dd7a0febe805d3a120b6
3,2,2,2,3,2,2,2,4,3,...,3,4,5,3,5,5,4,3,3,SinglePredator_LessInstructions_63dad9e2dd58781d74bfa642
4,2,1,2,1,1,1,1,1,1,...,2,1,2,3,5,4,2,1,1,SinglePredator_LessInstructions_5eb41fcb3a7a1724207bbfd6
5,1,1,2,1,1,2,1,1,1,...,2,2,3,3,5,1,2,1,2,SinglePredator_LessInstructions_610c59ee34cc312d7ca55e45


[1] 758

[1] 90

X,g,F1.,F2.,ID,q_item,Subscale_y
0,0.35,0.40,0.00,AI02_01,STAI,anxiety
1,0.38,0.00,0.27,AI02_02,STAI,anxiety
2,0.35,0.30,0.00,AI02_03,STAI,depression
3,0.34,0.39,0.00,AI02_04,STAI,depression
4,0.37,0.31,0.00,AI02_05,STAI,depression
5,0.27,0.00,0.00,AI02_06,STAI,depression
6,0.26,0.00,0.00,AI02_08,STAI,anxiety
7,0.34,0.00,0.24,AI02_09,STAI,anxiety
8,0.29,0.32,0.00,AI02_10,STAI,depression
9,0.34,0.20,0.00,AI02_12,STAI,anxiety


In [6]:
# Create a dataframe from loaded data

Q_data_sp_df = data.frame(Qns_data_common_predator)
Q_data_sp_df=na.omit(Q_data_sp_df)
Q_data_df_pure = subset(Q_data_sp_df,select = -c(X,REF))

head(Qns_data_common_predator)
nrow(Qns_data_common_predator)
nrow(Q_data_sp_df)
ncol(Q_data_sp_df)

X,AI02_01,AI02_02,AI02_03,AI02_04,AI02_05,AI02_06,AI02_08,AI02_09,AI02_10,...,PW01_04,PW01_07,PW01_08,PW01_09,PW01_10,PW01_12,PW01_13,PW01_15,PW01_16,REF
0,2,3,3,4,2,3,3,2,3,...,4,2,4,3,5,2,3,2,3,SinglePredator_LessInstructions_63e590489911159364dcf70d
1,3,2,3,3,2,3,2,2,3,...,4,4,3,3,5,5,4,4,3,SinglePredator_LessInstructions_5ea3325a58e7b70dd3026a42
2,2,2,2,3,2,3,2,3,2,...,3,3,5,2,5,4,3,3,3,SinglePredator_LessInstructions_5e88dd7a0febe805d3a120b6
3,2,2,2,3,2,2,2,4,3,...,3,4,5,3,5,5,4,3,3,SinglePredator_LessInstructions_63dad9e2dd58781d74bfa642
4,2,1,2,1,1,1,1,1,1,...,2,1,2,3,5,4,2,1,1,SinglePredator_LessInstructions_5eb41fcb3a7a1724207bbfd6
5,1,1,2,1,1,2,1,1,1,...,2,2,3,3,5,1,2,1,2,SinglePredator_LessInstructions_610c59ee34cc312d7ca55e45


[1] 758

[1] 758

[1] 90

In [ ]:
### Factor score function 
#- taken from 'psych packages'

In [7]:
"factor.scores" <- function(x,f,Phi=NULL,method=c("Thurstone","tenBerge","Anderson","Bartlett","Harman","components"),rho=NULL,impute="none") {
#the normal case is f is the structure matrix and Phi is not specified
#Note that the Grice formulas distinguish between Pattern and Structure matrices
#I need to confirm that I am doing this

    if(length(method) > 1) method <- "tenBerge"   #the default
    if(method=="regression") method <- "Thurstone"
    if(method=="tenberge") method <- "tenBerge"
    if(length(class(f)) > 1) { if(inherits(f[2] ,"irt.fa" )) f <- f$fa  }
    
     if(!is.matrix(f)) {Phi <- f$Phi
     f <- loadings(f)
      if(ncol(f)==1) {method <- "Thurstone"}
      }
     nf <- dim(f)[2]
      if(is.null(Phi)) Phi <- diag(1,nf,nf)
     if(dim(x)[1] == dim(f)[1]) {r <- as.matrix(x)
         square <- TRUE} else { 
          square <- FALSE
         if(!is.null(rho)) {r <- rho } else {
          r <- cor(x,use="pairwise") #find the correlation matrix from the data
      }}
      
      S <- f %*% Phi   #the Structure matrix 
   switch(method,   
    "Thurstone" = { w <- try(solve(r,S),silent=TRUE )  #these are the factor weights (see Grice eq. 5)
     	if(inherits(w,"try-error")) {message("In factor.scores, the correlation matrix is singular, an approximation is used")
               r <- cor.smooth(r)}
        
      w <- try(solve(r,S),silent=TRUE)
      if(inherits(w,"try-error")) {message("I was unable to calculate the factor score weights, factor loadings used instead")
               w <- f}
      colnames(w) <- colnames(f)
      rownames(w) <- rownames(f)
       }, 
      
  "tenBerge" = { #Following Grice equation 8 to estimate scores for oblique solutions (with a correction to the second line where r should r.inv
        L <- f %*% matSqrt(Phi)
        r.5 <- invMatSqrt(r)
       
        r <- cor.smooth(r)
        inv.r <- try(solve(r),silent=TRUE)
        if(inherits(inv.r, as.character("try-error")))  {warning("The tenBerge based scoring could not invert the correlation matrix, regression scores found instead")
                                                      ev <- eigen(r)
      ev$values[ev$values < .Machine$double.eps] <- 100 * .Machine$double.eps
        r <- ev$vectors %*% diag(ev$values) %*% t(ev$vectors)
        diag(r)  <- 1
       w <- solve(r,f)}  else {
        C <- r.5 %*% L %*% invMatSqrt(t(L) %*% inv.r %*% L)    #note that this is the correct formula, per Grice personal communication
        w <- r.5 %*% C %*% matSqrt(Phi)}
        colnames(w) <- colnames(f)
        rownames(w) <- rownames(f)
        },

 
       
    "Harman" = { #Grice equation 10 -- 
     #   m <- t(f)  %*% f  #factor intercorrelations 
     m <- f %*% t(S)  #should be this  (the model matrix)  Revised August 31, 2017
     diag(m) <- 1  #Grice does not say this, but it is necessary to make it work!
       inv.m <- solve(m)
     #  w <- f %*%inv.m  
     w <- inv.m %*% f
       }, 
       
       
        
    "Anderson" =  { #scores for orthogonal factor solution will be orthogonal  Grice Eq 7 and 8
    I <- diag(1,nf,nf)
    h2 <-  diag( f %*% Phi %*% t(f))
    U2 <- 1 - h2
    inv.U2 <- diag(1/U2)
    w <- inv.U2 %*% f %*% invMatSqrt(t(f) %*% inv.U2 %*% r %*% inv.U2 %*% f)
    colnames(w) <- colnames(f)
    rownames(w) <- rownames(f)
    },
    
   "Bartlett" = {    #Grice eq 9  # f should be the pattern, not the structure 
    I <- diag(1,nf,nf)
    h2 <-  diag( f %*% Phi %*% t(f))
    U2 <- 1 - h2
    inv.U2 <- diag(1/U2)
    w <- inv.U2 %*% f %*% (solve(t(f) %*% inv.U2 %*% f))
    colnames(w) <- colnames(f)
    rownames(w) <- rownames(f)
    },
    "none" = {w <- NULL},
    
    "components" = {w <- try(solve(r,f),silent=TRUE )    #basically, just do the regression/Thurstone approach for components
                    w <- f }
    )
    
    
    #now find a few fit statistics
    if(is.null(w)) {results <- list(scores=NULL,weights=NULL)} else {
     R2 <- diag(t(w) %*% S)  #this had been   R2 <- diag(t(w) %*% f)   Corrected Sept 1, 2017
     if(any(R2 > 1) || (prod(!is.nan(R2)) <1) || (prod(R2) < 0) ) {#message("The matrix is probably singular -- Factor score estimate results are likely incorrect")
                      R2[abs(R2) > 1] <- NA
                      R2[R2 <= 0] <- NA
                     }
     #if ((max(R2,na.rm=TRUE) > (1 + .Machine$double.eps)) ) {message("The estimated weights for the factor scores are probably incorrect.  Try a different factor extraction method.")}
      r.scores <- cov2cor(t(w) %*% r %*% w) #what actually is this?
     
    
  if(square) {  #that is, if given the correlation matrix
     class(w) <- NULL
     results <- list(scores=NULL,weights=w)
      results$r.scores <- r.scores 
   	  results$R2 <- R2   #this is the multiple R2 of the scores with the factors
     } else {
         missing <- rowSums(is.na(x))
    if(impute !="none") {
       x <- data.matrix(x)
        miss <- which(is.na(x),arr.ind=TRUE)
        if(impute=="mean") {
       		item.means <- colMeans(x,na.rm=TRUE)   #replace missing values with means
       		x[miss]<- item.means[miss[,2]]} else { 
       		item.med   <- apply(x,2,median,na.rm=TRUE) #replace missing with medians
        	x[miss]<- item.med[miss[,2]]}   #this only works if items is a matrix
     }
      

     if(method !="components") {scores <- x %*% w } else {  #standardize the data before doing the regression if using factors, 
        scores <- x %*% w}       # for components, the data have already been zero centered and, if appropriate, scaled
     results <- list(scores=scores,weights=w)
     results$r.scores <- r.scores
     results$missing <- missing 
   	  results$R2 <- R2   #this is the multiple R2 of the scores with the factors
     }
     }
   
     return(results) }
     #how to treat missing data?  see score.item
         
     
"matSqrt" <- function(x) {
   e <- eigen(x)
    e$values[e$values < 0] <- .Machine$double.eps
   sqrt.ev <- sqrt(e$values)   #need to put in a check here for postive semi definite
   result <- e$vectors %*% diag(sqrt.ev) %*% t(e$vectors)
   result}
   
   
"invMatSqrt" <- function(x) {
   e <- eigen(x)
   if(is.complex(e$values)) {warning("complex eigen values detected by invMatSqrt, results are suspect")
                 result <- x
      } else {
      
       e$values[e$values < .Machine$double.eps] <- 100 * .Machine$double.eps
   inv.sqrt.ev <- 1/sqrt(e$values)   #need to put in a check here for postive semi definite
   result <- e$vectors %*% diag(inv.sqrt.ev) %*% t(e$vectors) }
   result}

In [8]:
 # Scale qns data
Q_data_df_scaled_scale = scale(Q_data_df_pure)
head(Q_data_df_scaled_scale)
nrow(Q_data_df_pure)
ncol(Q_data_df_pure)
Q_data_df_scaled_scale

AI02_01,AI02_02,AI02_03,AI02_04,AI02_05,AI02_06,AI02_08,AI02_09,AI02_10,AI02_12,...,PW01_02,PW01_04,PW01_07,PW01_08,PW01_09,PW01_10,PW01_12,PW01_13,PW01_15,PW01_16
-0.7292915,0.7632817,0.1649714,1.3354212,-0.1511251,0.1446674,0.8373149,-0.4599129,0.4979330,-1.0256463,...,-0.08857696,0.6423863,-0.84850068,0.2393485,0.04626777,0.5770345,-0.8296685,-0.2777688,-0.6313571,-0.221055
0.5357038,-0.3689444,0.1649714,0.3420461,-0.1511251,0.1446674,-0.2152302,-0.4599129,0.4979330,0.1364462,...,0.73021978,0.6423863,0.69756546,-0.6499955,0.04626777,0.5770345,1.2527445,0.5414870,0.8807082,-0.221055
-0.7292915,-0.3689444,-0.9416510,0.3420461,-0.1511251,0.1446674,-0.2152302,0.5869759,-0.6852432,-1.0256463,...,0.73021978,-0.1773592,-0.07546761,1.1286926,-0.78875527,0.5770345,0.5586068,-0.2777688,0.1246756,-0.221055
-0.7292915,-0.3689444,-0.9416510,0.3420461,-0.1511251,-1.0472662,-0.2152302,1.6338647,0.4979330,0.1364462,...,0.73021978,-0.1773592,0.69756546,1.1286926,0.04626777,0.5770345,1.2527445,0.5414870,0.1246756,-0.221055
-0.7292915,-1.5011704,-0.9416510,-1.6447042,-1.1386491,-2.2391998,-1.2677753,-1.5068017,-1.8684193,-1.0256463,...,-0.90737369,-0.9971047,-1.62153375,-1.5393396,0.04626777,0.5770345,0.5586068,-1.0970247,-1.3873898,-1.880062
-1.9942868,-1.5011704,-0.9416510,-1.6447042,-1.1386491,-1.0472662,-1.2677753,-1.5068017,-1.8684193,-1.0256463,...,-0.90737369,-0.9971047,-0.84850068,-0.6499955,0.04626777,0.5770345,-1.5238062,-1.0970247,-1.3873898,-1.050559


[1] 758

[1] 88

AI02_01,AI02_02,AI02_03,AI02_04,AI02_05,AI02_06,AI02_08,AI02_09,AI02_10,AI02_12,...,PW01_02,PW01_04,PW01_07,PW01_08,PW01_09,PW01_10,PW01_12,PW01_13,PW01_15,PW01_16
-0.7292915,0.7632817,0.1649714,1.3354212,-0.1511251,0.1446674,0.8373149,-0.4599129,0.4979330,-1.0256463,...,-0.08857696,0.6423863,-0.84850068,0.2393485,0.04626777,0.5770345,-0.8296685,-0.2777688,-0.6313571,-0.2210550
0.5357038,-0.3689444,0.1649714,0.3420461,-0.1511251,0.1446674,-0.2152302,-0.4599129,0.4979330,0.1364462,...,0.73021978,0.6423863,0.69756546,-0.6499955,0.04626777,0.5770345,1.2527445,0.5414870,0.8807082,-0.2210550
-0.7292915,-0.3689444,-0.9416510,0.3420461,-0.1511251,0.1446674,-0.2152302,0.5869759,-0.6852432,-1.0256463,...,0.73021978,-0.1773592,-0.07546761,1.1286926,-0.78875527,0.5770345,0.5586068,-0.2777688,0.1246756,-0.2210550
-0.7292915,-0.3689444,-0.9416510,0.3420461,-0.1511251,-1.0472662,-0.2152302,1.6338647,0.4979330,0.1364462,...,0.73021978,-0.1773592,0.69756546,1.1286926,0.04626777,0.5770345,1.2527445,0.5414870,0.1246756,-0.2210550
-0.7292915,-1.5011704,-0.9416510,-1.6447042,-1.1386491,-2.2391998,-1.2677753,-1.5068017,-1.8684193,-1.0256463,...,-0.90737369,-0.9971047,-1.62153375,-1.5393396,0.04626777,0.5770345,0.5586068,-1.0970247,-1.3873898,-1.8800622
-1.9942868,-1.5011704,-0.9416510,-1.6447042,-1.1386491,-1.0472662,-1.2677753,-1.5068017,-1.8684193,-1.0256463,...,-0.90737369,-0.9971047,-0.84850068,-0.6499955,0.04626777,0.5770345,-1.5238062,-1.0970247,-1.3873898,-1.0505586
-0.7292915,-0.3689444,-0.9416510,-0.6513291,-0.1511251,-1.0472662,-0.2152302,-0.4599129,-0.6852432,0.1364462,...,-0.08857696,-0.1773592,-0.84850068,-0.6499955,0.04626777,-0.5620076,-0.1355308,0.5414870,-0.6313571,-1.0505586
0.5357038,-0.3689444,0.1649714,-0.6513291,0.8363990,-1.0472662,-0.2152302,0.5869759,0.4979330,1.2985388,...,-0.90737369,-0.1773592,-0.07546761,-0.6499955,0.04626777,-1.7010497,-0.1355308,-0.2777688,0.1246756,-0.2210550
-0.7292915,-0.3689444,0.1649714,0.3420461,0.8363990,-1.0472662,-1.2677753,-0.4599129,-1.8684193,0.1364462,...,-0.08857696,0.6423863,-0.07546761,0.2393485,0.88129080,0.5770345,0.5586068,-0.2777688,-0.6313571,0.6084485
0.5357038,-0.3689444,1.2715939,0.3420461,0.8363990,0.1446674,0.8373149,0.5869759,0.4979330,1.2985388,...,0.73021978,0.6423863,0.69756546,0.2393485,0.88129080,0.5770345,1.2527445,-0.2777688,0.8807082,0.6084485


In [9]:
# Get loadings for each item of questionnaire
L2.cdm = Gagne_loadings[,c(2,3,4)]
Lmat2.cdm = as.matrix(L2.cdm)
head(Lmat2.cdm)
Lmat2.cdm

g,F1.,F2.
0.35,0.40,0.00
0.38,0.00,0.27
0.35,0.30,0.00
0.34,0.39,0.00
0.37,0.31,0.00
0.27,0.00,0.00


g,F1.,F2.
0.35,0.40,0.00
0.38,0.00,0.27
0.35,0.30,0.00
0.34,0.39,0.00
0.37,0.31,0.00
0.27,0.00,0.00
0.26,0.00,0.00
0.34,0.00,0.24
0.29,0.32,0.00
0.34,0.20,0.00


In [10]:
# calculate factor scores
fscores<-factor.scores(Q_data_df_scaled_scale,
                       Lmat2.cdm,method ="Anderson",impute = 'mean')
head(fscores$scores)
fscores2.cdm<-fscores

g,F1.,F2.
-0.2423816,1.47962068,-0.4583042
-0.4701803,1.31683994,0.9543438
-0.7312739,-0.26727338,0.8916696
-0.7553792,-0.56271327,1.8424877
-1.4786477,0.02680913,-0.1216633
-0.9062970,-1.29447462,-0.5407669


In [11]:
factor_scores_subj = data.frame(fscores$scores)
factor_scores_subj['V1'] = Q_data_sp_df$REF
factor_scores_subj

g,F1.,F2.,V1
-0.2423816,1.47962068,-0.45830424,SinglePredator_LessInstructions_63e590489911159364dcf70d
-0.4701803,1.31683994,0.95434382,SinglePredator_LessInstructions_5ea3325a58e7b70dd3026a42
-0.7312739,-0.26727338,0.89166965,SinglePredator_LessInstructions_5e88dd7a0febe805d3a120b6
-0.7553792,-0.56271327,1.84248775,SinglePredator_LessInstructions_63dad9e2dd58781d74bfa642
-1.4786477,0.02680913,-0.12166325,SinglePredator_LessInstructions_5eb41fcb3a7a1724207bbfd6
-0.9062970,-1.29447462,-0.54076688,SinglePredator_LessInstructions_610c59ee34cc312d7ca55e45
-0.9227781,0.15876608,0.33114919,SinglePredator_LessInstructions_5fb70808c7c7bb1d76c2b9d4
-0.5443262,0.94614383,0.19580191,SinglePredator_LessInstructions_6538827212ccfc55c37f985d
-0.1558231,-0.08031836,0.68127880,SinglePredator_LessInstructions_5fc8384d1bc5841b0e23dce6
1.4608714,0.21864568,-0.46256506,SinglePredator_LessInstructions_6321d9c78e8e3c04f82a0ea0


In [12]:
# save factor scores

if (clinical_study == 1) {
    write.csv(factor_scores_subj,'./supplementary_data/factor_analysis/Predator_FS_using_GagneClinicalLoadings.csv')
    
} else { 
    write.csv(factor_scores_subj,'./supplementary_data/factor_analysis/Predator_FS_using_GagneConfLoadings.csv')
  }

